# NeMo Customizer Service Test

This notebook tests the NeMo Customizer service to verify it's working correctly.

## Prerequisites
- NeMo Customizer service deployed and running
- NeMo Data Store service deployed and running
- NeMo Entity Store service deployed and running
- Access to the cluster namespace

## Step 1: Install Required Packages

In [ ]:
%pip install requests python-dotenv huggingface_hub

## Step 2: Import Libraries and Configure URLs

In [ ]:
import os
import json
import time
import requests
from pathlib import Path
from pprint import pprint

# Load environment variables from env.donotcommit if it exists
try:
    from dotenv import load_dotenv
    # In Jupyter notebooks, use current working directory (where notebook is run from)
    # The env.donotcommit file should be in the same directory as the notebook
    env_donotcommit_path = Path.cwd() / "env.donotcommit"
    if env_donotcommit_path.exists():
        load_dotenv(env_donotcommit_path, override=False)
        print(f"✅ Loaded env.donotcommit from: {env_donotcommit_path}")
    else:
        print(f"ℹ️  env.donotcommit not found at: {env_donotcommit_path}")
        print(f"   Using environment variables from system/defaults")
except ImportError:
    print("⚠️  python-dotenv not installed - using system environment variables only")

# Namespace configuration
NMS_NAMESPACE = os.getenv("NMS_NAMESPACE", "anemo-rhoai")

# Service URLs (cluster-internal)
CUSTOMIZER_URL = f"http://nemocustomizer-sample.{NMS_NAMESPACE}.svc.cluster.local:8000"
DATASTORE_URL = f"http://nemodatastore-sample.{NMS_NAMESPACE}.svc.cluster.local:8000"
ENTITY_STORE_URL = f"http://nemoentitystore-sample.{NMS_NAMESPACE}.svc.cluster.local:8000"

# API Keys and Tokens (from env.donotcommit)
NGC_API_KEY = os.getenv("NGC_API_KEY", "")
HF_TOKEN = os.getenv("HF_TOKEN", "")
NDS_TOKEN = os.getenv("NDS_TOKEN", "token")
NIM_SERVICE_ACCOUNT_TOKEN = os.getenv("NIM_SERVICE_ACCOUNT_TOKEN", "")

# Inference Service Configuration (for testing models)
INFERENCE_SERVICE_URL = os.getenv("INFERENCE_SERVICE_URL", "")
INFERENCE_SERVICE_NAME = os.getenv("INFERENCE_SERVICE_NAME", "")
BASE_MODEL = os.getenv("BASE_MODEL", "meta/llama-3.2-1b-instruct")
CUSTOMIZATION_TEMPLATE = os.getenv("CUSTOMIZATION_TEMPLATE", "meta/llama-3.2-1b-instruct@v1.0.0")

# Set Hugging Face token for dataset access
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HF_ENDPOINT"] = "https://huggingface.co"
    print("✅ Hugging Face token configured")
else:
    print("⚠️ HF_TOKEN not set - Hugging Face dataset access may be limited")

print(f"Customizer URL: {CUSTOMIZER_URL}")
print(f"DataStore URL: {DATASTORE_URL}")
print(f"Entity Store URL: {ENTITY_STORE_URL}")
print(f"Namespace: {NMS_NAMESPACE}")

## Step 3: Test Customizer Service Health

In [ ]:
# Test health endpoint
try:
    health_url = f"{CUSTOMIZER_URL}/v1/health/ready"
    print(f"Checking health at: {health_url}")
    response = requests.get(health_url, timeout=10)
    print(f"Status Code: {response.status_code}")
    if response.status_code == 200:
        health_data = response.json()
        print("✅ Customizer service is healthy!")
        print(f"Response: {health_data}")
    else:
        print(f"⚠️ Unexpected status code: {response.status_code}")
        print(f"Response: {response.text}")
except requests.exceptions.RequestException as e:
    print(f"❌ Error connecting to customizer service: {e}")
    print("Please verify:")
    print(f"  1. Customizer service is running: oc get pods -n {NMS_NAMESPACE} | grep customizer")
    print(f"  2. Service exists: oc get svc nemocustomizer-sample -n {NMS_NAMESPACE}")
    print(f"  3. You're running this notebook from within the cluster (Workbench/Notebook)")

## Step 4: List Customization Jobs (API Verification)

In [ ]:
# List existing customization jobs (this verifies the API is working)
try:
    jobs_url = f"{CUSTOMIZER_URL}/v1/customization/jobs"
    print(f"Listing jobs from: {jobs_url}")
    response = requests.get(jobs_url, timeout=10)
    print(f"Status Code: {response.status_code}")
    if response.status_code == 200:
        jobs = response.json()
        job_list = jobs.get('data', [])
        print(f"✅ API is working! Found {len(job_list)} customization job(s):")
        if job_list:
            for job in job_list[:5]:  # Show first 5 jobs
                print(f"  - Job ID: {job.get('id')}, Status: {job.get('status')}, Created: {job.get('created_at')}")
            if len(job_list) > 5:
                print(f"  ... and {len(job_list) - 5} more jobs")
        else:
            print("  No jobs found (this is normal for a fresh deployment)")
        print(f"\nTotal jobs: {jobs.get('pagination', {}).get('total_results', len(job_list))}")
    else:
        print(f"⚠️ Unexpected status code: {response.status_code}")
        print(f"Response: {response.text}")
except requests.exceptions.RequestException as e:
    print(f"❌ Error listing jobs: {e}")

## Step 5: Test DataStore Connection (Required for Customization)

## Step 6: Test DataStore Connection (Required for Customization)

In [ ]:
# Test DataStore health (required for customization)
try:
    datastore_health_url = f"{DATASTORE_URL}/v1/health"
    print(f"Checking DataStore health at: {datastore_health_url}")
    response = requests.get(datastore_health_url, timeout=10)
    print(f"Status Code: {response.status_code}")
    if response.status_code == 200:
        health_data = response.json()
        print("✅ DataStore service is healthy!")
        print(f"Status: {health_data.get('status')}")
        print(f"Description: {health_data.get('description')}")
        checks = health_data.get('checks', {})
        if checks:
            print("Checks:")
            for check_name, check_results in checks.items():
                for result in check_results:
                    print(f"  - {check_name}: {result.get('status')}")
    else:
        print(f"⚠️ Unexpected status code: {response.status_code}")
        print(f"Response: {response.text}")
except requests.exceptions.RequestException as e:
    print(f"❌ Error connecting to DataStore service: {e}")
    print("Please verify DataStore is running:")
    print(f"  oc get pods -n {NMS_NAMESPACE} | grep datastore")

## Step 7: Test Entity Store Connection (Required for Model Registration)

In [ ]:
# Test Entity Store health (required for model registration)
try:
    entity_store_health_url = f"{ENTITY_STORE_URL}/health"
    print(f"Checking Entity Store health at: {entity_store_health_url}")
    response = requests.get(entity_store_health_url, timeout=10)
    print(f"Status Code: {response.status_code}")
    if response.status_code == 200:
        health_data = response.json()
        print("✅ Entity Store service is healthy!")
        print(f"Status: {health_data.get('status')}")
    else:
        print(f"⚠️ Unexpected status code: {response.status_code}")
        print(f"Response: {response.text}")
except requests.exceptions.RequestException as e:
    print(f"❌ Error connecting to Entity Store service: {e}")
    print("Please verify Entity Store is running:")
    print(f"  oc get pods -n {NMS_NAMESPACE} | grep entitystore")

## Step 8: Test DataStore Upload (Create Namespace)

In [ ]:
# Test DataStore namespace creation (required for dataset uploads)
try:
    namespace_url = f"{DATASTORE_URL}/v1/datastore/namespaces/{NMS_NAMESPACE}"
    print(f"Checking namespace at: {namespace_url}")
    response = requests.get(
        namespace_url, 
        headers={"Authorization": f"Bearer {NDS_TOKEN}"},
        timeout=10
    )
    
    if response.status_code == 404:
        # Create namespace
        print(f"Creating namespace: {NMS_NAMESPACE}")
        response = requests.post(
            f"{DATASTORE_URL}/v1/datastore/namespaces",
            json={"name": NMS_NAMESPACE},
            headers={"Authorization": f"Bearer {NDS_TOKEN}"},
            timeout=10
        )
        if response.status_code in (200, 201):
            print(f"✅ Created namespace: {NMS_NAMESPACE}")
        else:
            print(f"⚠️ Failed to create namespace: {response.status_code} - {response.text}")
    elif response.status_code == 200:
        print(f"✅ Namespace exists: {NMS_NAMESPACE}")
    else:
        print(f"⚠️ Unexpected status code: {response.status_code} - {response.text}")
except requests.exceptions.RequestException as e:
    print(f"❌ Error with DataStore namespace: {e}")

## Step 9: Test DataStore Upload (Sample Dataset)

In [ ]:
# Test uploading a sample dataset to DataStore
# This uses the HuggingFace-compatible API that DataStore provides
import json
import tempfile

# Initialize variables (will be used in later cells)
TEST_DATASET_NAME = "customizer-test-dataset"
repo_id = f"{NMS_NAMESPACE}/{TEST_DATASET_NAME}"
ENTITY_STORE_DATASET_ID = None  # Will be set in Step 10

try:
    from huggingface_hub import HfApi
    
    # Initialize HuggingFace API pointing to DataStore
    hf_endpoint = f"{DATASTORE_URL}/v1/hf"
    hf_token = NDS_TOKEN if NDS_TOKEN != "token" else None
    hf_api = HfApi(endpoint=hf_endpoint, token=hf_token)
    
    # IMPORTANT: Ensure namespace exists in Gitea before creating repository
    # Data Store's Gitea backend needs the namespace directory to exist,
    # otherwise it defaults to "default" namespace and commits fail.
    # Creating a temporary repository first ensures the namespace is created in Gitea.
    print(f"Step 1: Initializing namespace in DataStore (if needed)...")
    temp_repo_id = f"{NMS_NAMESPACE}/namespace-init-temp"
    try:
        # Create temporary repo to ensure namespace exists in Gitea
        hf_api.create_repo(repo_id=temp_repo_id, repo_type="dataset", exist_ok=True)
        # Delete temporary repo (namespace directory will remain)
        try:
            hf_api.delete_repo(repo_id=temp_repo_id, repo_type="dataset")
        except:
            pass  # Ignore if deletion fails
        print(f"✅ Namespace '{NMS_NAMESPACE}' initialized in Gitea")
    except Exception as e:
        # If temp repo creation fails, namespace might already exist - continue
        print(f"ℹ️  Namespace check: {e}")
    
    print(f"\nStep 2: Creating dataset repo in DataStore: {repo_id}")
    
    # Smart approach: Check if repo exists and is accessible
    # If corrupted (500 error), delete and recreate. If good, reuse it.
    repo_needs_creation = True
    try:
        check_url = f"{DATASTORE_URL}/v1/hf/api/datasets/{repo_id}/revision/main"
        check_response = requests.get(check_url, timeout=10)
        if check_response.status_code == 200:
            print(f"✅ Repo exists and is accessible - will reuse it")
            repo_needs_creation = False
        elif check_response.status_code == 500:
            print(f"⚠️  Repo exists but is corrupted (500 error) - will delete and recreate")
            try:
                hf_api.delete_repo(repo_id=repo_id, repo_type="dataset")
                print(f"   ✅ Deleted corrupted repo")
            except Exception as e:
                print(f"   ℹ️  Delete attempt: {e}")
        else:
            print(f"ℹ️  Repo doesn't exist ({check_response.status_code}) - will create it")
    except Exception as e:
        print(f"ℹ️  Repo check failed (assuming doesn't exist): {e}")
    
    # Create the repo if needed
    if repo_needs_creation:
        try:
            hf_api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)
            print(f"✅ Created dataset repo: {repo_id}")
            
            # IMPORTANT: Wait for git repository initialization to complete
            # Gitea needs time to initialize the git directory structure
            import time
            print(f"   ⏳ Waiting 3s for git repository initialization...")
            time.sleep(3)
            print(f"✅ Repository ready for uploads")
        except Exception as e:
            if "already exists" in str(e).lower():
                print(f"ℹ️ Dataset repo already exists: {repo_id}")
                # Still wait for git init in case it was just created
                import time
                time.sleep(3)
            else:
                print(f"⚠️ Error creating repo: {e}")
                raise
    
    # Create a sample training file
    print(f"\nStep 3: Creating sample training data...")
    sample_data = {
        "messages": [
            {"role": "user", "content": "What is the capital of France?"},
            {"role": "assistant", "content": "The capital of France is Paris."}
        ]
    }
    
    # Create temporary file
    with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
        json.dump(sample_data, f, indent=2)
        temp_file = f.name
    
    try:
        print(f"\nStep 4: Uploading sample file to DataStore...")
        # Upload file - this will trigger a commit operation
        upload_succeeded = False
        max_upload_retries = 2
        
        for upload_attempt in range(max_upload_retries):
            try:
                hf_api.upload_file(
                    path_or_fileobj=temp_file,
                    path_in_repo="training/sample.json",
                    repo_id=repo_id,
                    repo_type="dataset"
                )
                print(f"✅ Uploaded sample file to DataStore")
                print(f"   Dataset location: hf://datasets/{repo_id}")
                upload_succeeded = True
                break
            except Exception as upload_error:
                error_str = str(upload_error).lower()
                # Check if it's a 500 error (corrupted repo) and we can retry
                if ("500" in error_str or "internal server error" in error_str) and upload_attempt < max_upload_retries - 1:
                    print(f"⚠️  Upload failed with 500 error (attempt {upload_attempt + 1}/{max_upload_retries})")
                    print(f"   Repository may have become corrupted - will delete and recreate...")
                    try:
                        hf_api.delete_repo(repo_id=repo_id, repo_type="dataset")
                        print(f"   ✅ Deleted corrupted repo")
                    except:
                        pass
                    
                    # Recreate with namespace init
                    temp_repo_id = f"{NMS_NAMESPACE}/namespace-init-temp"
                    try:
                        hf_api.create_repo(repo_id=temp_repo_id, repo_type="dataset", exist_ok=True)
                        hf_api.delete_repo(repo_id=temp_repo_id, repo_type="dataset")
                    except:
                        pass
                    
                    hf_api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)
                    print(f"   ✅ Recreated repo")
                    import time
                    time.sleep(3)
                    print(f"   ✅ Waited for git init - retrying upload...")
                else:
                    # Not a retryable error or last attempt
                    raise
        
        if not upload_succeeded:
            raise Exception("Upload failed after retries")
        
        # Verify upload completed by checking if repository is accessible
        print(f"\nStep 5: Verifying upload completed...")
        import time
        max_retries = 5
        retry_delay = 2
        
        for attempt in range(max_retries):
            try:
                # Check if we can access the repository (what Customizer will do)
                verify_url = f"{DATASTORE_URL}/v1/hf/api/datasets/{repo_id}/revision/main"
                verify_response = requests.get(verify_url, timeout=10)
                
                if verify_response.status_code == 200:
                    print(f"✅ Verified: Dataset repository is accessible")
                    break
                elif verify_response.status_code == 500:
                    if attempt < max_retries - 1:
                        print(f"   ⏳ Repository not ready yet, waiting {retry_delay}s... (attempt {attempt + 1}/{max_retries})")
                        time.sleep(retry_delay)
                    else:
                        print(f"⚠️  Warning: Repository verification failed after {max_retries} attempts")
                        print(f"   The commit may have failed. Check DataStore logs if upload issues persist.")
                else:
                    print(f"⚠️  Unexpected status code: {verify_response.status_code}")
                    break
            except Exception as verify_error:
                if attempt < max_retries - 1:
                    print(f"   ⏳ Verification failed, retrying in {retry_delay}s... (attempt {attempt + 1}/{max_retries})")
                    time.sleep(retry_delay)
                else:
                    print(f"⚠️  Verification error: {verify_error}")
    finally:
        # Clean up temp file
        import os
        if os.path.exists(temp_file):
            os.unlink(temp_file)
    
    print(f"\n✅ Dataset upload test successful!")
    print(f"   Dataset can be accessed via: hf://datasets/{repo_id}")
    
except ImportError:
    print("❌ huggingface_hub not installed - install with: pip install huggingface_hub")
except Exception as e:
    print(f"❌ Error uploading dataset: {e}")
    import traceback
    traceback.print_exc()

## Step 10: Test Entity Store Dataset Registration

In [ ]:
# Test registering the dataset in Entity Store
# IMPORTANT: Customizer needs datasets registered in Entity Store (not just DataStore)
# The Entity Store provides the dataset ID that Customizer uses

try:
    files_url = f"hf://datasets/{repo_id}"
    print(f"Registering dataset in Entity Store...")
    print(f"   Dataset name: {TEST_DATASET_NAME}")
    print(f"   Files URL: {files_url}")
    
    response = requests.post(
        f"{ENTITY_STORE_URL}/v1/datasets",
        json={
            "name": TEST_DATASET_NAME,
            "namespace": NMS_NAMESPACE,
            "description": "Test dataset for Customizer service verification",
            "files_url": files_url,
            "project": "customizer-test",
            "format": "json",
        },
        timeout=30
    )
    
    if response.status_code in (200, 201):
        dataset_obj = response.json()
        print(f"✅ Dataset registered in Entity Store!")
        print(f"   Dataset ID: {dataset_obj.get('id', 'N/A')}")
        print(f"   Files URL: {dataset_obj.get('files_url', 'N/A')}")
        ENTITY_STORE_DATASET_ID = dataset_obj.get('id')
    elif response.status_code == 409:
        print(f"ℹ️ Dataset {TEST_DATASET_NAME} already exists in Entity Store")
        # Try to get the existing dataset
        get_response = requests.get(
            f"{ENTITY_STORE_URL}/v1/datasets/{NMS_NAMESPACE}/{TEST_DATASET_NAME}",
            timeout=30
        )
        if get_response.status_code == 200:
            dataset_obj = get_response.json()
            ENTITY_STORE_DATASET_ID = dataset_obj.get('id')
            print(f"   Dataset ID: {ENTITY_STORE_DATASET_ID}")
            print(f"   Files URL: {dataset_obj.get('files_url', 'N/A')}")
    else:
        print(f"⚠️ Failed to register in Entity Store: {response.status_code}")
        print(f"   Response: {response.text}")
        ENTITY_STORE_DATASET_ID = None
        
except requests.exceptions.RequestException as e:
    print(f"❌ Error registering dataset in Entity Store: {e}")
    ENTITY_STORE_DATASET_ID = None

## Step 11: Verify Customizer Can Find the Dataset

In [ ]:
# Verify that Customizer can access the dataset via Entity Store
# Customizer uses Entity Store to resolve dataset IDs to actual file locations

if ENTITY_STORE_DATASET_ID:
    print(f"Verifying Customizer can access dataset via Entity Store...")
    print(f"   Entity Store Dataset ID: {ENTITY_STORE_DATASET_ID}")
    
    # Check if Entity Store has the dataset
    try:
        get_response = requests.get(
            f"{ENTITY_STORE_URL}/v1/datasets/{NMS_NAMESPACE}/{TEST_DATASET_NAME}",
            timeout=30
        )
        if get_response.status_code == 200:
            dataset_info = get_response.json()
            print(f"✅ Entity Store has the dataset")
            print(f"   Name: {dataset_info.get('name')}")
            print(f"   Files URL: {dataset_info.get('files_url')}")
            print(f"   Format: {dataset_info.get('format')}")
            
            # Verify the files_url points to DataStore
            files_url = dataset_info.get('files_url', '')
            if files_url.startswith('hf://datasets/'):
                print(f"✅ Dataset files URL is correctly formatted for DataStore")
                print(f"   Customizer can use this dataset ID: {ENTITY_STORE_DATASET_ID}")
            else:
                print(f"⚠️ Unexpected files URL format: {files_url}")
        else:
            print(f"⚠️ Could not retrieve dataset from Entity Store: {get_response.status_code}")
    except Exception as e:
        print(f"❌ Error verifying dataset: {e}")
else:
    print("⚠️ Skipping verification - dataset not registered in Entity Store")

## Step 12: Test Base Model (Before Customization)

Before customizing, let's test the base model to see its responses. This will help us verify that customization actually changed the model's behavior.

In [ ]:
# Test the base model with sample prompts (before customization)
# This helps verify that customization actually changed the model's behavior

if not INFERENCE_SERVICE_URL:
    print("⚠️  INFERENCE_SERVICE_URL not set in env.donotcommit")
    print("   Skipping base model test - set INFERENCE_SERVICE_URL to test model responses")
    print("   Example: INFERENCE_SERVICE_URL=http://your-inference-service:8000")
    base_responses = {}
else:
    # Disable SSL warnings for HTTPS
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    
    # Headers for InferenceService
    headers = {
        "Content-Type": "application/json"
    }
    if NIM_SERVICE_ACCOUNT_TOKEN:
        headers["Authorization"] = f"Bearer {NIM_SERVICE_ACCOUNT_TOKEN}"
    
    # Detect which URL works by testing with a simple request
    # Build list of potential URLs to test
    potential_urls = [INFERENCE_SERVICE_URL]
    if INFERENCE_SERVICE_NAME:
        # Add internal URL if we have service name
        if INFERENCE_SERVICE_URL.startswith('https://'):
            internal_url = f"https://{INFERENCE_SERVICE_NAME}-predictor.{NMS_NAMESPACE}.svc.cluster.local:8443"
        else:
            internal_url = f"http://{INFERENCE_SERVICE_NAME}-predictor.{NMS_NAMESPACE}.svc.cluster.local:80"
        potential_urls.append(internal_url)
    
    # Determine model name to use for testing
    test_model_name = INFERENCE_SERVICE_NAME if INFERENCE_SERVICE_NAME else (BASE_MODEL if BASE_MODEL else None)
    
    # Test URLs to find one that works
    working_url = None
    print("🔍 Detecting working InferenceService URL...")
    for test_url in potential_urls:
        if not test_model_name:
            continue
        try:
            test_payload = {
                "model": test_model_name,
                "messages": [{"role": "user", "content": "test"}],
                "max_tokens": 5
            }
            response = requests.post(
                f"{test_url}/v1/chat/completions",
                headers=headers,
                json=test_payload,
                timeout=10,
                verify=False
            )
            if response.status_code == 200:
                working_url = test_url
                print(f"✅ Found working URL: {test_url}")
                break
            elif response.status_code == 503:
                print(f"⚠️  {test_url} returned 503 (service unavailable), trying next...")
            else:
                print(f"⚠️  {test_url} returned {response.status_code}, trying next...")
        except Exception as e:
            print(f"⚠️  {test_url} failed: {str(e)[:50]}, trying next...")
            continue
    
    if not working_url:
        print("❌ No working URL found. Cannot test base model.")
        print("   Please check:")
        print(f"   1. InferenceService is running: oc get inferenceservice {INFERENCE_SERVICE_NAME} -n {NMS_NAMESPACE}")
        print(f"   2. Pods are ready: oc get pods -n {NMS_NAMESPACE} | grep {INFERENCE_SERVICE_NAME}")
        print(f"   3. Service account token is valid")
        base_responses = {}
    else:
        # Sample prompts to test
        test_prompts = [
            "What is machine learning?",
            "Explain quantum computing in simple terms.",
            "Write a haiku about technology."
        ]
        
        # Store base model responses
        base_responses = {}
        
        print(f"\nTesting base model responses using: {working_url}")
        print("=" * 60)
        
        # vLLM uses the InferenceService name as the served model name (--served-model-name)
        # So we use INFERENCE_SERVICE_NAME as the model name in API requests
        # Fallback to BASE_MODEL if INFERENCE_SERVICE_NAME is not set
        model_names_to_try = []
        if INFERENCE_SERVICE_NAME:
            model_names_to_try.append(INFERENCE_SERVICE_NAME)  # Primary: use InferenceService name (e.g., anemo-rhoai-model-ssr)
        if BASE_MODEL:
            model_names_to_try.append(BASE_MODEL)  # Fallback: try BASE_MODEL (e.g., meta/llama-3.2-1b-instruct)
            # Also try variations of BASE_MODEL as additional fallbacks
            if "meta/" in BASE_MODEL:
                model_names_to_try.append(BASE_MODEL.replace("meta/", ""))  # Without prefix
                model_names_to_try.append(f"models/{BASE_MODEL.replace('meta/', '')}")  # With models/ prefix
        # Remove duplicates while preserving order
        model_names_to_try = list(dict.fromkeys(model_names_to_try))
        
        if not model_names_to_try:
            print("⚠️  No model name available - set INFERENCE_SERVICE_NAME or BASE_MODEL in env.donotcommit")
            base_responses = {}
        else:
            for prompt in test_prompts:
                print(f"\n📝 Prompt: {prompt}")
                
                try:
                    # Use OpenAI-compatible API endpoint
                    success = False
                    for model_name in model_names_to_try:
                        if success:
                            break
                            
                        payload = {
                            "model": model_name,
                            "messages": [
                                {"role": "user", "content": prompt}
                            ],
                            "temperature": 0.7,
                            "max_tokens": 150
                        }
                        
                        try:
                            response = requests.post(
                                f"{working_url}/v1/chat/completions",
                                headers=headers,
                                json=payload,
                                timeout=30,
                                verify=False  # Disable SSL verification for HTTPS
                            )
                            
                            if response.status_code == 200:
                                result = response.json()
                                answer = result["choices"][0]["message"]["content"]
                                base_responses[prompt] = answer
                                print(f"✅ Success with model '{model_name}'!")
                                print(f"✅ Response: {answer[:100]}...")
                                success = True
                                break  # Success, no need to try other model names
                            elif response.status_code == 404:
                                # Model not found - try next model name format
                                error_msg = response.text[:200]
                                if "does not exist" in error_msg or "not found" in error_msg.lower():
                                    print(f"   ⚠️  Model '{model_name}' not found, trying next format...")
                                    break  # Try next model name
                                else:
                                    print(f"   ❌ Error 404: {error_msg}")
                                    if model_name == model_names_to_try[-1]:  # Last attempt
                                        base_responses[prompt] = None
                                    continue
                            else:
                                print(f"   ❌ Error {response.status_code}: {response.text[:200]}")
                                if model_name == model_names_to_try[-1]:  # Last attempt
                                    base_responses[prompt] = None
                                continue  # Try next model name
                        except requests.exceptions.ConnectionError as ce:
                            print(f"   ❌ Connection error: {ce}")
                            if model_name == model_names_to_try[-1]:  # Last attempt
                                base_responses[prompt] = None
                            continue  # Try next model name
                        except Exception as e:
                            print(f"   ❌ Exception: {e}")
                            if model_name == model_names_to_try[-1]:  # Last attempt
                                base_responses[prompt] = None
                            continue  # Try next model name
                    
                    if not success:
                        base_responses[prompt] = None
                        print(f"   ❌ Failed with all model name formats: {model_names_to_try}")
                        
                except Exception as e:
                    print(f"❌ Exception: {e}")
                    base_responses[prompt] = None
            
            print("\n" + "=" * 60)
            print(f"✅ Base model testing complete. Saved {len([r for r in base_responses.values() if r])} responses.")
        
        print("\n" + "=" * 60)
        print(f"✅ Base model testing complete. Saved {len([r for r in base_responses.values() if r])} responses.")

## Step 13: Prepare Training Data for Customization

Now let's prepare training data in the correct format for customization. We'll use OpenAI format (messages) as required by NeMo Customizer.

In [ ]:
# Prepare training data for customization
# IMPORTANT: Must use OpenAI format (messages) as per NVIDIA reference
# IMPORTANT: Customizer requires BOTH training AND validation files
# Reference: https://github.com/NVIDIA/k8s-nim-operator/blob/main/test/e2e/jupyter-notebook/e2e-notebook.ipynb

# Use TEST_DATASET_NAME from Step 9 if available, otherwise use default
if 'TEST_DATASET_NAME' not in locals():
    TEST_DATASET_NAME = "customizer-test-dataset"
CUSTOMIZATION_DATASET_NAME = f"{TEST_DATASET_NAME}-customization"

# IMPORTANT: Need enough examples to split into training and validation sets
# Customizer's train_val_split function requires at least 2 files (one for train, one for val)
# We'll create 6 examples and split 80/20 (5 train, 1 val)
all_training_data = [
    {
        "messages": [
            {"role": "user", "content": "What is machine learning?"},
            {"role": "assistant", "content": "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It uses algorithms to analyze data, identify patterns, and make predictions or decisions."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain quantum computing in simple terms."},
            {"role": "assistant", "content": "Quantum computing uses quantum mechanical phenomena like superposition and entanglement to perform computations. Unlike classical computers that use bits (0 or 1), quantum computers use quantum bits or qubits that can exist in multiple states simultaneously, allowing them to solve certain problems much faster."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Write a haiku about technology."},
            {"role": "assistant", "content": "Silicon dreams flow,\nCircuits hum with endless light,\nFuture's code unfolds."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is artificial intelligence?"},
            {"role": "assistant", "content": "Artificial intelligence (AI) is the simulation of human intelligence in machines that are programmed to think and learn like humans. It encompasses machine learning, natural language processing, computer vision, and robotics to create systems capable of performing tasks that typically require human intelligence."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How does neural network training work?"},
            {"role": "assistant", "content": "Neural network training involves feeding data through the network, comparing predictions to actual outcomes, calculating errors, and then adjusting the network's weights and biases through backpropagation. This iterative process continues until the network learns to make accurate predictions on the training data."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is deep learning?"},
            {"role": "assistant", "content": "Deep learning is a subset of machine learning that uses neural networks with multiple layers (hence 'deep') to learn hierarchical representations of data. These deep neural networks can automatically discover complex patterns in large datasets, making them powerful for tasks like image recognition, natural language processing, and speech recognition."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a transformer in AI?"},
            {"role": "assistant", "content": "A transformer is a deep learning architecture that uses attention mechanisms to process sequences of data. Unlike recurrent neural networks, transformers can process all parts of a sequence in parallel, making them highly efficient. They form the basis of modern language models like GPT and BERT."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain reinforcement learning."},
            {"role": "assistant", "content": "Reinforcement learning is a type of machine learning where an agent learns to make decisions by interacting with an environment. The agent receives rewards or penalties for its actions and learns to maximize cumulative rewards over time. It's used in game playing, robotics, and autonomous systems."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is natural language processing?"},
            {"role": "assistant", "content": "Natural language processing (NLP) is a branch of AI that focuses on enabling computers to understand, interpret, and generate human language. It combines computational linguistics with machine learning to process text and speech, enabling applications like translation, sentiment analysis, and chatbots."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is computer vision?"},
            {"role": "assistant", "content": "Computer vision is a field of AI that enables machines to interpret and understand visual information from the world. It involves processing, analyzing, and understanding digital images and videos to extract meaningful information, enabling applications like facial recognition, object detection, and medical image analysis."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How does gradient descent work?"},
            {"role": "assistant", "content": "Gradient descent is an optimization algorithm used to minimize a cost function by iteratively moving in the direction of steepest descent. It calculates the gradient (derivative) of the cost function with respect to model parameters and updates them in small steps, gradually converging to the optimal values."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is overfitting in machine learning?"},
            {"role": "assistant", "content": "Overfitting occurs when a machine learning model learns the training data too well, including noise and irrelevant patterns, resulting in poor performance on new, unseen data. It happens when the model is too complex relative to the amount of training data, leading to high variance and low generalization."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain supervised learning."},
            {"role": "assistant", "content": "Supervised learning is a type of machine learning where the algorithm learns from labeled training data. The model is trained on input-output pairs, learning to map inputs to correct outputs. Common tasks include classification (predicting categories) and regression (predicting continuous values)."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is unsupervised learning?"},
            {"role": "assistant", "content": "Unsupervised learning is a type of machine learning where the algorithm finds patterns in data without labeled examples. The model explores the data structure on its own, identifying hidden patterns, clusters, or relationships. Common tasks include clustering, dimensionality reduction, and anomaly detection."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a neural network?"},
            {"role": "assistant", "content": "A neural network is a computing system inspired by biological neural networks. It consists of interconnected nodes (neurons) organized in layers that process information. Each connection has a weight that adjusts during training, allowing the network to learn complex patterns and relationships in data."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain backpropagation."},
            {"role": "assistant", "content": "Backpropagation is an algorithm used to train neural networks by calculating gradients of the loss function with respect to network weights. It works by propagating error signals backward through the network, from output to input layers, enabling efficient weight updates that minimize prediction errors."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a convolutional neural network?"},
            {"role": "assistant", "content": "A convolutional neural network (CNN) is a type of deep learning architecture designed for processing grid-like data such as images. It uses convolutional layers with filters that detect features like edges and patterns, making it highly effective for image recognition, object detection, and computer vision tasks."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is transfer learning?"},
            {"role": "assistant", "content": "Transfer learning is a technique where a model trained on one task is reused or adapted for a related task. Instead of training from scratch, you start with a pre-trained model and fine-tune it on your specific dataset, saving time and computational resources while often achieving better performance."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain regularization in machine learning."},
            {"role": "assistant", "content": "Regularization is a technique used to prevent overfitting by adding constraints or penalties to the model. Common methods include L1 and L2 regularization, which add penalty terms to the loss function based on model complexity, encouraging simpler models that generalize better to new data."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a loss function?"},
            {"role": "assistant", "content": "A loss function measures how well a machine learning model's predictions match the actual target values. It quantifies the error between predicted and true values, providing a single number that the training process aims to minimize. Common examples include mean squared error for regression and cross-entropy for classification."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is feature engineering?"},
            {"role": "assistant", "content": "Feature engineering is the process of selecting, transforming, and creating input variables (features) to improve model performance. It involves domain knowledge to extract meaningful information from raw data, such as creating new features, scaling values, or encoding categorical variables."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain cross-validation."},
            {"role": "assistant", "content": "Cross-validation is a technique for assessing model performance by splitting data into multiple folds. The model is trained on some folds and tested on others, repeating this process to get a more reliable estimate of generalization performance and reduce the impact of data splitting variability."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a hyperparameter?"},
            {"role": "assistant", "content": "A hyperparameter is a configuration setting that controls the learning process but is not learned from data. Examples include learning rate, number of layers, batch size, and regularization strength. Hyperparameters are set before training and significantly impact model performance."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is batch normalization?"},
            {"role": "assistant", "content": "Batch normalization is a technique that normalizes the inputs to each layer by adjusting and scaling activations. It helps stabilize training, allows higher learning rates, and can improve model performance by reducing internal covariate shift during training."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain dropout in neural networks."},
            {"role": "assistant", "content": "Dropout is a regularization technique that randomly sets a fraction of neurons to zero during training. This prevents the network from becoming too dependent on specific neurons, reducing overfitting and improving generalization by forcing the network to learn more robust features."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is an activation function?"},
            {"role": "assistant", "content": "An activation function determines the output of a neural network node given its input. It introduces non-linearity, enabling the network to learn complex patterns. Common examples include ReLU (rectified linear unit), sigmoid, and tanh, each with different properties for various use cases."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is fine-tuning?"},
            {"role": "assistant", "content": "Fine-tuning is the process of taking a pre-trained model and adapting it to a specific task by continuing training on a new dataset. Instead of training from scratch, you start with learned features and adjust them for your particular use case, which is faster and often more effective."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain the bias-variance tradeoff."},
            {"role": "assistant", "content": "The bias-variance tradeoff is a fundamental concept in machine learning describing the tension between model complexity and generalization. High bias (underfitting) occurs with simple models that miss patterns, while high variance (overfitting) occurs with complex models that capture noise. The goal is finding the right balance."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a learning rate?"},
            {"role": "assistant", "content": "The learning rate is a hyperparameter that controls how much the model weights are updated during training. A high learning rate can cause unstable training, while a low learning rate may result in slow convergence. Finding the right learning rate is crucial for effective model training."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is data augmentation?"},
            {"role": "assistant", "content": "Data augmentation is a technique that artificially increases the size of a training dataset by applying transformations like rotation, scaling, or cropping to existing examples. This helps improve model generalization and reduce overfitting, especially when working with limited training data."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain ensemble methods."},
            {"role": "assistant", "content": "Ensemble methods combine multiple machine learning models to make predictions, often achieving better performance than individual models. Common approaches include bagging (training models on different data subsets), boosting (sequentially improving weak models), and stacking (combining predictions from diverse models)."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a confusion matrix?"},
            {"role": "assistant", "content": "A confusion matrix is a table used to evaluate classification model performance. It shows the counts of true positives, true negatives, false positives, and false negatives, allowing calculation of metrics like accuracy, precision, recall, and F1 score to assess how well the model performs on different classes."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is precision and recall?"},
            {"role": "assistant", "content": "Precision measures the accuracy of positive predictions (true positives / (true positives + false positives)), while recall measures the ability to find all positive instances (true positives / (true positives + false negatives)). They are often in tension - improving one may decrease the other."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain the F1 score."},
            {"role": "assistant", "content": "The F1 score is the harmonic mean of precision and recall, providing a single metric that balances both concerns. It's calculated as 2 * (precision * recall) / (precision + recall), giving equal weight to precision and recall, making it useful when you need to balance false positives and false negatives."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a ROC curve?"},
            {"role": "assistant", "content": "A ROC (Receiver Operating Characteristic) curve plots the true positive rate against the false positive rate at various classification thresholds. The area under the ROC curve (AUC-ROC) provides a single metric for model performance, with values closer to 1.0 indicating better classification ability."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is early stopping?"},
            {"role": "assistant", "content": "Early stopping is a regularization technique that halts training when validation performance stops improving. It prevents overfitting by stopping before the model memorizes training data, saving computational resources and often resulting in better generalization to new data."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain gradient clipping."},
            {"role": "assistant", "content": "Gradient clipping is a technique that limits the magnitude of gradients during training to prevent exploding gradients. It caps gradient values at a threshold, ensuring stable training in deep networks and preventing weight updates from becoming too large and destabilizing the learning process."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a learning curve?"},
            {"role": "assistant", "content": "A learning curve plots model performance (like loss or accuracy) against training iterations or dataset size. It helps diagnose issues like overfitting (training performance much better than validation) or underfitting (both are poor), guiding decisions about model complexity and training duration."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a recurrent neural network?"},
            {"role": "assistant", "content": "A recurrent neural network (RNN) is a type of neural network designed for sequential data, with connections that form cycles allowing information to persist. RNNs process sequences step by step, maintaining hidden states that capture information from previous steps, making them suitable for tasks like language modeling and time series prediction."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain LSTM networks."},
            {"role": "assistant", "content": "LSTM (Long Short-Term Memory) networks are a type of RNN designed to address the vanishing gradient problem. They use gating mechanisms (forget, input, and output gates) to selectively remember or forget information over long sequences, making them effective for tasks requiring long-term dependencies."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is attention mechanism?"},
            {"role": "assistant", "content": "Attention mechanism allows a model to focus on relevant parts of the input when making predictions. Instead of processing all inputs equally, it learns to weight different parts based on their importance, enabling better handling of long sequences and improving performance in tasks like translation and summarization."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a GAN?"},
            {"role": "assistant", "content": "A GAN (Generative Adversarial Network) consists of two neural networks - a generator that creates fake data and a discriminator that tries to distinguish real from fake. They train together in an adversarial process, with the generator improving its ability to create realistic data that fools the discriminator."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain word embeddings."},
            {"role": "assistant", "content": "Word embeddings are dense vector representations of words that capture semantic relationships. Words with similar meanings are positioned close together in the embedding space, enabling models to understand relationships like synonyms, analogies, and context. Popular methods include Word2Vec, GloVe, and contextual embeddings from transformers."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is tokenization?"},
            {"role": "assistant", "content": "Tokenization is the process of breaking text into smaller units called tokens, which can be words, subwords, or characters. It's a crucial preprocessing step for NLP models, converting raw text into a format that neural networks can process, with different strategies balancing vocabulary size and handling of rare words."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is BERT?"},
            {"role": "assistant", "content": "BERT (Bidirectional Encoder Representations from Transformers) is a transformer-based language model that processes text bidirectionally, understanding context from both left and right. Pre-trained on large text corpora, BERT can be fine-tuned for various NLP tasks like question answering, sentiment analysis, and named entity recognition."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain GPT models."},
            {"role": "assistant", "content": "GPT (Generative Pre-trained Transformer) models are autoregressive language models that generate text by predicting the next word in a sequence. They're pre-trained on vast amounts of text and can be fine-tuned or used with prompts for tasks like text generation, summarization, and question answering."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is few-shot learning?"},
            {"role": "assistant", "content": "Few-shot learning is a machine learning approach where a model learns to perform a task with very few training examples. It leverages prior knowledge from pre-training to quickly adapt to new tasks, often using techniques like meta-learning or prompt engineering to achieve good performance with minimal data."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is zero-shot learning?"},
            {"role": "assistant", "content": "Zero-shot learning is the ability of a model to perform a task without any task-specific training examples. The model uses its general knowledge from pre-training to understand and complete tasks it hasn't explicitly been trained on, often through natural language instructions or prompts."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain model distillation."},
            {"role": "assistant", "content": "Model distillation is a technique where a smaller, faster student model learns to mimic a larger, more accurate teacher model. The student is trained on the teacher's predictions (soft labels) rather than hard labels, often achieving similar performance with much lower computational requirements."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is federated learning?"},
            {"role": "assistant", "content": "Federated learning is a distributed machine learning approach where models are trained across multiple devices or servers without centralizing data. Each participant trains on local data and shares only model updates, preserving privacy while enabling collaborative learning from decentralized data sources."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is explainable AI?"},
            {"role": "assistant", "content": "Explainable AI (XAI) refers to methods and techniques that make AI model decisions understandable to humans. It includes techniques like feature importance, attention visualization, and model interpretability methods that help users understand why a model made a particular prediction, increasing trust and enabling debugging."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is MLOps?"},
            {"role": "assistant", "content": "MLOps (Machine Learning Operations) is a set of practices that combines machine learning with DevOps principles. It focuses on automating and streamlining the ML lifecycle, including model development, deployment, monitoring, and maintenance, ensuring models can be reliably deployed and updated in production environments."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a data pipeline?"},
            {"role": "assistant", "content": "A data pipeline is an automated process that moves and transforms data from source to destination. It includes steps like data extraction, cleaning, transformation, validation, and loading, ensuring data flows efficiently and reliably through systems for analysis, training, or serving machine learning models."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain A/B testing in ML."},
            {"role": "assistant", "content": "A/B testing in ML compares two versions of a model or system to determine which performs better. Users are randomly split into groups, each experiencing a different version, and metrics are collected to statistically determine which version provides better results, enabling data-driven decisions about model deployment."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is model versioning?"},
            {"role": "assistant", "content": "Model versioning is the practice of tracking different versions of machine learning models throughout their lifecycle. It includes versioning model code, training data, hyperparameters, and weights, enabling reproducibility, rollback to previous versions, and comparison of model performance across different iterations."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is continuous learning?"},
            {"role": "assistant", "content": "Continuous learning (also called online learning or incremental learning) is the ability of a model to learn from new data over time without retraining from scratch. The model updates its knowledge incrementally as new data arrives, adapting to changing patterns and maintaining relevance in dynamic environments."}
        ]
    }
]

# Split into training (80%) and validation (20%) sets
# IMPORTANT: Validation set must have at least as many samples as batch_size (8)
# With 50 examples and 80/20 split: 40 train, 10 val (validation has enough for batch_size 8)
import math
split_idx = math.floor(len(all_training_data) * 0.8)  # Use floor to ensure validation gets enough
training_data = all_training_data[:split_idx]
validation_data = all_training_data[split_idx:]

# Ensure validation has at least 8 samples (matching batch_size)
if len(validation_data) < 8:
    # Move samples from training to validation if needed
    while len(validation_data) < 8 and len(training_data) > 0:
        validation_data.append(training_data.pop())

print(f"✅ Prepared {len(all_training_data)} total examples")
print(f"   Training examples: {len(training_data)}")
print(f"   Validation examples: {len(validation_data)}")
print("\nSample training example:")
print(json.dumps(training_data[0], indent=2))
print("\nSample validation example:")
print(json.dumps(validation_data[0], indent=2))

## Step 14: Upload Training Data to DataStore

Upload the training data to DataStore using the HuggingFace-compatible API.

In [ ]:
# Upload training data to DataStore
import tempfile

customization_repo_id = f"{NMS_NAMESPACE}/{CUSTOMIZATION_DATASET_NAME}"

try:
    from huggingface_hub import HfApi
    
    # Initialize HuggingFace API pointing to DataStore
    hf_endpoint = f"{DATASTORE_URL}/v1/hf"
    hf_token = NDS_TOKEN if NDS_TOKEN != "token" else None
    hf_api = HfApi(endpoint=hf_endpoint, token=hf_token)
    
    # IMPORTANT: Ensure namespace exists in Gitea before creating repository
    # Data Store's Gitea backend needs the namespace directory to exist,
    # otherwise it defaults to "default" namespace. Creating a temporary
    # repository first ensures the namespace is created in Gitea.
    print(f"Step 1: Initializing namespace in DataStore (if needed)...")
    temp_repo_id = f"{NMS_NAMESPACE}/namespace-init-temp"
    try:
        # Create temporary repo to ensure namespace exists in Gitea
        hf_api.create_repo(repo_id=temp_repo_id, repo_type="dataset", exist_ok=True)
        # Delete temporary repo (namespace directory will remain)
        try:
            hf_api.delete_repo(repo_id=temp_repo_id, repo_type="dataset")
        except:
            pass  # Ignore if deletion fails
        print(f"✅ Namespace '{NMS_NAMESPACE}' initialized in Gitea")
    except Exception as e:
        # If temp repo creation fails, namespace might already exist - continue
        print(f"ℹ️  Namespace check: {e}")
    
    print(f"\nStep 2: Creating/updating dataset repo in DataStore: {customization_repo_id}")
    
    # IMPORTANT: Delete existing repo to ensure clean structure
    # The dataset may have mixed file locations (root + training/) which confuses Customizer
    # We'll recreate it with the correct structure (files in training/ subdirectory)
    repo_needs_creation = True
    try:
        check_url = f"{DATASTORE_URL}/v1/hf/api/datasets/{customization_repo_id}/revision/main"
        check_response = requests.get(check_url, timeout=10)
        if check_response.status_code == 200:
            print(f"⚠️  Repo exists - deleting to recreate with correct structure")
            try:
                hf_api.delete_repo(repo_id=customization_repo_id, repo_type="dataset")
                print(f"   ✅ Deleted existing repo (will recreate with training/val.jsonl)")
                import time
                time.sleep(2)  # Brief delay before recreation
            except Exception as e:
                print(f"   ⚠️  Delete attempt: {e} (will try to create anyway)")
        elif check_response.status_code == 500:
            print(f"⚠️  Repo exists but is corrupted (500 error) - will delete and recreate")
            try:
                hf_api.delete_repo(repo_id=customization_repo_id, repo_type="dataset")
                print(f"   ✅ Deleted corrupted repo")
                import time
                time.sleep(2)
            except Exception as e:
                print(f"   ℹ️  Delete attempt: {e}")
        else:
            print(f"ℹ️  Repo doesn't exist ({check_response.status_code}) - will create it")
    except Exception as e:
        print(f"ℹ️  Repo check failed (assuming doesn't exist): {e}")
    
    # Always create the repo (fresh start with correct structure)
    try:
        hf_api.create_repo(repo_id=customization_repo_id, repo_type="dataset", exist_ok=True)
        print(f"✅ Created dataset repo: {customization_repo_id}")
        
        # IMPORTANT: Wait for git repository initialization to complete
        # Gitea needs time to initialize the git directory structure
        import time
        print(f"   ⏳ Waiting 3s for git repository initialization...")
        time.sleep(3)
        print(f"✅ Repository ready for uploads")
    except Exception as e:
        if "already exists" in str(e).lower():
            print(f"ℹ️ Dataset repo already exists: {customization_repo_id}")
            # Still wait for git init in case it was just created
            import time
            time.sleep(3)
        else:
            print(f"⚠️ Error creating repo: {e}")
            raise
    
    # IMPORTANT: Customizer requires BOTH training AND validation files
    # Create JSONL files (one JSON object per line) for both training and validation
    print(f"\nStep 3: Creating training and validation data files...")
    
    # Create training file
    with tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=False) as f:
        for item in training_data:
            f.write(json.dumps(item) + '\n')
        temp_train_file = f.name
    
    # Create validation file
    with tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=False) as f:
        for item in validation_data:
            f.write(json.dumps(item) + '\n')
        temp_val_file = f.name
    
    try:
        print(f"\nStep 4: Uploading training and validation data to DataStore...")
        upload_succeeded = False
        max_upload_retries = 2
        
        for upload_attempt in range(max_upload_retries):
            try:
                # NeMo Customizer 25.x (25.08, 25.12+) requires: training/train.jsonl and validation/validation.jsonl
                # Tested locally - confirmed this structure works for all 25.x versions
                hf_api.upload_file(
                    path_or_fileobj=temp_train_file,
                    path_in_repo="training/train.jsonl",
                    repo_id=customization_repo_id,
                    repo_type="dataset"
                )
                print(f"✅ Uploaded: training/train.jsonl ({len(training_data)} samples)")
                
                hf_api.upload_file(
                    path_or_fileobj=temp_val_file,
                    path_in_repo="validation/validation.jsonl",
                    repo_id=customization_repo_id,
                    repo_type="dataset"
                )
                print(f"✅ Uploaded: validation/validation.jsonl ({len(validation_data)} samples)")
                
                print(f"\n✅ Uploaded both training and validation data to DataStore")
                print(f"   Dataset location: hf://datasets/{customization_repo_id}")
                print(f"   Training samples: {len(training_data)}")
                print(f"   Validation samples: {len(validation_data)}")
                upload_succeeded = True
                break
            except Exception as upload_error:
                error_str = str(upload_error).lower()
                # Check if it's a 500 error (corrupted repo) and we can retry
                if ("500" in error_str or "internal server error" in error_str) and upload_attempt < max_upload_retries - 1:
                    print(f"⚠️  Upload failed with 500 error (attempt {upload_attempt + 1}/{max_upload_retries})")
                    print(f"   Repository may have become corrupted - will delete and recreate...")
                    try:
                        hf_api.delete_repo(repo_id=customization_repo_id, repo_type="dataset")
                        print(f"   ✅ Deleted corrupted repo")
                    except:
                        pass
                    
                    # Recreate with namespace init
                    temp_repo_id = f"{NMS_NAMESPACE}/namespace-init-temp"
                    try:
                        hf_api.create_repo(repo_id=temp_repo_id, repo_type="dataset", exist_ok=True)
                        hf_api.delete_repo(repo_id=temp_repo_id, repo_type="dataset")
                    except:
                        pass
                    
                    hf_api.create_repo(repo_id=customization_repo_id, repo_type="dataset", exist_ok=True)
                    print(f"   ✅ Recreated repo")
                    import time
                    time.sleep(3)
                    print(f"   ✅ Waited for git init - retrying upload...")
                else:
                    # Not a retryable error or last attempt
                    raise
        
        if not upload_succeeded:
            raise Exception("Upload failed after retries")
        
        # Step 5: Verify upload completed by listing repository files
        # This ensures the commit succeeded and repository is accessible
        print(f"\nStep 5: Verifying upload completed...")
        import time
        max_retries = 5
        retry_delay = 2
        
        for attempt in range(max_retries):
            try:
                repo_files = hf_api.list_repo_files(repo_id=customization_repo_id, repo_type="dataset")
                if repo_files:
                    print(f"✅ Verified: Repository contains {len(repo_files)} file(s)")
                    for file in repo_files:
                        print(f"   - {file}")
                    break
                else:
                    if attempt < max_retries - 1:
                        print(f"   ⏳ Repository empty, waiting {retry_delay}s for commit to complete... (attempt {attempt + 1}/{max_retries})")
                        time.sleep(retry_delay)
                    else:
                        raise Exception("Repository is empty after upload - commit may have failed!")
            except Exception as verify_error:
                if attempt < max_retries - 1:
                    print(f"   ⏳ Verification failed, retrying in {retry_delay}s... (attempt {attempt + 1}/{max_retries})")
                    print(f"   Error: {str(verify_error)[:100]}")
                    time.sleep(retry_delay)
                else:
                    print(f"❌ Upload verification failed after {max_retries} attempts: {verify_error}")
                    print(f"   The customization job will fail if the dataset is not accessible!")
                    raise
    finally:
        # Clean up temp files
        import os
        if 'temp_train_file' in locals() and os.path.exists(temp_train_file):
            os.unlink(temp_train_file)
        if 'temp_val_file' in locals() and os.path.exists(temp_val_file):
            os.unlink(temp_val_file)
    
    print(f"\n✅ Training data upload and verification successful!")
    
except ImportError:
    print("❌ huggingface_hub not installed - install with: pip install huggingface_hub")
except Exception as e:
    print(f"❌ Error uploading training data: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Helper: Generate unique output model name with retry logic for job creation
# This helps avoid "model already exists" errors from DataStore

import random

def create_customization_job_with_retry(customizer_url, training_params_template, max_retries=3):
    """
    Create a customization job with automatic retry if model repo already exists.
    
    Args:
        customizer_url: Base URL for Customizer service
        training_params_template: Template dict for training params (will be modified)
        max_retries: Maximum number of retry attempts
    
    Returns:
        tuple: (job_id, job_response) or (None, None) if failed
    """
    headers_custom = {"Content-Type": "application/json"}
    
    for attempt in range(max_retries):
        # Generate unique timestamp with random offset
        timestamp = int(time.time()) + random.randint(1, 10000)
        
        # IMPORTANT: Use NMS_NAMESPACE as the namespace (not "meta") to avoid conflicts
        # Extract model name from BASE_MODEL (e.g., "meta/llama-3.2-1b-instruct" -> "llama-3.2-1b-instruct")
        if "/" in BASE_MODEL:
            model_name_only = BASE_MODEL.split("/", 1)[1]  # Get part after namespace
        else:
            model_name_only = BASE_MODEL
        
        # Use NMS_NAMESPACE as namespace and add "-custom" suffix to distinguish from base model
        unique_output_model = f"{NMS_NAMESPACE}/{model_name_only}-custom@1.0-{timestamp}"
        
        # Update training params with new model name
        training_params = training_params_template.copy()
        training_params["output_model"] = unique_output_model
        training_params["name"] = f"customizer-test-job-{timestamp}"
        
        if attempt > 0:
            print(f"\n🔄 Retry {attempt + 1}/{max_retries}: Using output model: {unique_output_model}")
        else:
            print(f"\n📋 Using unique output model: {unique_output_model}")
        
        try:
            response = requests.post(
                f"{customizer_url}/v1/customization/jobs",
                json=training_params,
                headers=headers_custom,
                timeout=60
            )
            
            if response.status_code in (200, 201):
                job_response = response.json()
                job_id = job_response.get("id")
                return job_id, job_response
            elif response.status_code == 500:
                error_text = response.text.lower()
                if ("already exists" in error_text or "model or dataset" in error_text) and attempt < max_retries - 1:
                    print(f"⚠️  Model repo already exists, retrying with different timestamp...")
                    time.sleep(1)
                    continue
                else:
                    print(f"❌ Error: {response.text[:500]}")
                    return None, None
            else:
                print(f"❌ HTTP {response.status_code}: {response.text[:500]}")
                return None, None
                
        except Exception as e:
            print(f"❌ Exception: {e}")
            if attempt < max_retries - 1:
                time.sleep(1)
                continue
            return None, None
    
    return None, None

print("✅ Helper function loaded: create_customization_job_with_retry()")

## Step 15: Register Customization Dataset in Entity Store

Register the customization dataset in Entity Store so the Customizer can use it.

In [ ]:
# Register customization dataset in Entity Store
# IMPORTANT: Verify dataset exists in DataStore before registering in Entity Store
# Customizer validates datasets by calling DataStore API, so the repository must exist

try:
    files_url = f"hf://datasets/{customization_repo_id}"
    print(f"Step 1: Verifying dataset exists in DataStore...")
    print(f"   Dataset: {customization_repo_id}")
    
    # Verify dataset repository exists in DataStore before Entity Store registration
    # Customizer will call DataStore API to validate, so this must succeed
    import time
    max_verify_retries = 10
    verify_retry_delay = 3
    dataset_verified = False
    
    for verify_attempt in range(max_verify_retries):
        try:
            # Try to access the dataset revision endpoint (what Customizer checks)
            verify_url = f"{DATASTORE_URL}/v1/hf/api/datasets/{customization_repo_id}/revision/main"
            verify_response = requests.get(verify_url, timeout=10)
            
            if verify_response.status_code == 200:
                print(f"✅ Dataset verified in DataStore!")
                dataset_verified = True
                break
            elif verify_response.status_code == 500:
                if verify_attempt < max_verify_retries - 1:
                    print(f"   ⏳ Dataset repository not ready yet, waiting {verify_retry_delay}s... (attempt {verify_attempt + 1}/{max_verify_retries})")
                    print(f"   Error: Repository may not exist - commit may still be in progress")
                    time.sleep(verify_retry_delay)
                else:
                    print(f"❌ Dataset repository does not exist in DataStore after {max_verify_retries} attempts")
                    print(f"   This means the upload commit failed. Please re-run Step 14 (upload) cell.")
                    raise Exception(f"Dataset {customization_repo_id} not found in DataStore - upload may have failed")
            else:
                print(f"⚠️ Unexpected status code from DataStore: {verify_response.status_code}")
                print(f"   Response: {verify_response.text[:200]}")
                if verify_attempt < max_verify_retries - 1:
                    time.sleep(verify_retry_delay)
                else:
                    raise Exception(f"Could not verify dataset in DataStore: {verify_response.status_code}")
        except requests.exceptions.RequestException as verify_error:
            if verify_attempt < max_verify_retries - 1:
                print(f"   ⏳ Verification request failed, retrying in {verify_retry_delay}s... (attempt {verify_attempt + 1}/{max_verify_retries})")
                print(f"   Error: {str(verify_error)[:100]}")
                time.sleep(verify_retry_delay)
            else:
                raise Exception(f"Failed to verify dataset in DataStore: {verify_error}")
    
    if not dataset_verified:
        raise Exception("Dataset verification failed - cannot proceed with Entity Store registration")
    
    print(f"\nStep 2: Registering customization dataset in Entity Store...")
    print(f"   Dataset name: {CUSTOMIZATION_DATASET_NAME}")
    print(f"   Files URL: {files_url}")
    
    response = requests.post(
        f"{ENTITY_STORE_URL}/v1/datasets",
        json={
            "name": CUSTOMIZATION_DATASET_NAME,
            "namespace": NMS_NAMESPACE,
            "description": "Training data for model customization",
            "files_url": files_url,
            "project": "customizer-test",
            "format": "json",
        },
        timeout=30
    )
    
    if response.status_code in (200, 201):
        dataset_obj = response.json()
        print(f"✅ Customization dataset registered in Entity Store!")
        print(f"   Dataset ID: {dataset_obj.get('id', 'N/A')}")
        print(f"   Files URL: {dataset_obj.get('files_url', 'N/A')}")
        customization_dataset_registered = True
    elif response.status_code == 409:
        print(f"ℹ️ Dataset {CUSTOMIZATION_DATASET_NAME} already exists in Entity Store")
        customization_dataset_registered = True
    else:
        print(f"⚠️ Failed to register in Entity Store: {response.status_code}")
        print(f"   Response: {response.text}")
        customization_dataset_registered = False
        
except requests.exceptions.RequestException as e:
    print(f"❌ Error registering dataset in Entity Store: {e}")
    customization_dataset_registered = False
except Exception as e:
    print(f"❌ Error: {e}")
    print(f"\n💡 Troubleshooting:")
    print(f"   1. Re-run Step 14 (Upload Training Data to DataStore) cell")
    print(f"   2. Wait for the verification step to complete successfully")
    print(f"   3. Then re-run this cell (Step 15)")
    customization_dataset_registered = False

## Step 16: Create Customization Job

Create a customization job using the Customizer API to fine-tune the base model with our training data.

**⚠️ GPU Requirements:**
- Each training job requires 1 GPU
- Ensure GPUs are available before creating the job
- If all GPUs are busy, training pods will remain in `Pending` state
- To check GPU availability: `oc get nodes -o json | grep nvidia.com/gpu`
- To free up GPUs, scale down non-essential services: `oc scale deployment <service-name> -n <namespace> --replicas=0`

In [ ]:
# Create customization job using Customizer API
# This follows the official NVIDIA k8s-nim-operator e2e notebook implementation

if not customization_dataset_registered:
    print("❌ Cannot create customization job - dataset not registered")
    job_id = None
    job_response = None
else:
    print(f"✅ Prerequisites validated")
    print(f"  Base Model Config: {CUSTOMIZATION_TEMPLATE}")
    print(f"  Dataset Name: {CUSTOMIZATION_DATASET_NAME}")
    print(f"  Dataset Namespace: {NMS_NAMESPACE}")
    print(f"  Dataset URL: hf://datasets/{customization_repo_id}")
    
    # Generate unique output model name with timestamp to avoid conflicts
    # IMPORTANT: Use retry logic to handle "model already exists" errors
    import random
    max_retries = 3
    job_created = False
    job_id = None
    job_response = None
    
    for attempt in range(max_retries):
        timestamp = int(time.time())
        random_suffix = random.randint(10000, 99999)  # Add random component for uniqueness
        
        # Extract model name from BASE_MODEL (e.g., "meta/llama-3.2-1b-instruct" -> "llama-3.2-1b-instruct")
        if "/" in BASE_MODEL:
            model_name_only = BASE_MODEL.split("/", 1)[1]  # Get part after namespace
        else:
            model_name_only = BASE_MODEL
        
        # IMPORTANT: Include timestamp in model name (not just version) to avoid conflicts
        # Customizer checks if model name exists, so we need unique model names, not just unique versions
        # Format: namespace/model-name-custom-timestamp-random@version
        unique_output_model = f"{NMS_NAMESPACE}/{model_name_only}-custom-{timestamp}-{random_suffix}@1.0"
        
        if attempt > 0:
            print(f"\n🔄 Retry {attempt + 1}/{max_retries}: Using output model: {unique_output_model}")
        else:
            print(f"📋 Using unique output model: {unique_output_model}")
            print(f"   Namespace: {NMS_NAMESPACE} (avoids conflicts with 'meta' namespace)")
            print(f"   Timestamp + Random: {timestamp}-{random_suffix} (ensures uniqueness)")
        
        # IMPORTANT: Payload format based on tested API requirements
        # - config: Must be a STRING (model version), not a dict!
        # - dataset: Must be a STRING (hf://datasets/...), not a dict!
        # - hyperparameters: Must be a dict with all required fields
        training_params = {
            "base_model": BASE_MODEL,
            "dataset": f"hf://datasets/{customization_repo_id}",  # String, not dict!
            "output_model": unique_output_model,
            "template": CUSTOMIZATION_TEMPLATE,
            "config": f"{BASE_MODEL}@v1.0.0",  # String format: model@version
                            "hyperparameters": {
                    "finetuning_type": "lora",
                    "training_type": "sft",
                    "batch_size": 8,  # Keep at 8 - validation set will have enough samples
                    "epochs": 1,
                    "learning_rate": 0.0001,
                "lora": {
                    "adapter_dim": 16,
                    "alpha": 16,
                    "adapter_dropout": 0.1,
                    "target_modules": None
                },
                "sequence_packing_enabled": False
            }
        }
        
        if attempt == 0:
            print(f"\n📋 Request payload:")
            print(json.dumps(training_params, indent=2))
        
        try:
            if attempt == 0:
                print(f"\n🚀 Creating customization job...")
                print(f"   Customizer URL: {CUSTOMIZER_URL}")
                print(f"   Endpoint: /v1/customization/jobs")
            
            headers_custom = {"Content-Type": "application/json"}
            
            response = requests.post(
                f"{CUSTOMIZER_URL}/v1/customization/jobs",
                json=training_params,
                headers=headers_custom,
                timeout=60
            )
            
            if response.status_code in (200, 201):
                job_response = response.json()
                job_id = job_response.get("id")
                print(f"✅ Customization job created successfully!")
                print(f"   Job ID: {job_id}")
                print(f"   Output Model: {unique_output_model}")
                print(f"   Status: {job_response.get('status', 'N/A')}")
                print(f"\n💡 Next steps:")
                print(f"   1. The job will create a Volcano Job and training worker pod")
                print(f"   2. Training worker requires 1 GPU - ensure GPUs are available")
                print(f"   3. Check training pod: oc get pods -n {NMS_NAMESPACE} | grep {job_id}")
                print(f"   4. Check Volcano Job: oc get vj -n {NMS_NAMESPACE} | grep {job_id}")
                print(f"   5. If pod is Pending, check GPU availability or scale down other services")
                job_created = True
                break
            elif response.status_code == 500:
                error_text = response.text.lower()
                if "already exists" in error_text and attempt < max_retries - 1:
                    print(f"⚠️  Model repo already exists, retrying with different name...")
                    time.sleep(1)
                    continue
                else:
                    print(f"❌ Failed to create customization job (500 error)")
                    print(f"   Response: {response.text[:500]}")
                    if attempt == max_retries - 1:
                        job_id = None
                        job_response = None
            else:
                print(f"❌ Failed to create customization job")
                print(f"   HTTP Status: {response.status_code}")
                print(f"   Response: {response.text[:500]}")
                if attempt == max_retries - 1:
                    job_id = None
                    job_response = None
                    break
                    
        except Exception as e:
            print(f"❌ Exception creating customization job: {e}")
            if attempt < max_retries - 1:
                print(f"   Retrying with different model name...")
                time.sleep(1)
                continue
            else:
                import traceback
                traceback.print_exc()
                job_id = None
                job_response = None
    
    if not job_created:
        print(f"\n❌ Failed to create customization job after {max_retries} attempts")
        print(f"   Please check:")
        print(f"   1. Dataset exists in DataStore: {customization_repo_id}")
        print(f"   2. Dataset is registered in Entity Store")
        print(f"   3. Customizer service is running")
        job_id = None
        job_response = None

## Step 17: Wait for Customization Job to Complete

Monitor the customization job until it completes. This may take several minutes depending on the dataset size and model.

In [ ]:
# Wait for customization job to complete
def wait_customization_job(job_id: str, polling_interval: int = 30):
    """Wait for customization job to complete (runs indefinitely until job completes or fails)"""
    start_time = time.time()
    
    print(f"Waiting for customization job {job_id} to finish...")
    print(f"   Polling interval: {polling_interval} seconds")
    print(f"   ⚠️  Monitoring will continue indefinitely until job completes or fails")
    print(f"   💡 You can interrupt the kernel (Kernel -> Interrupt) if needed")
    
    try:
        # Get initial status
        response = requests.get(
            f"{CUSTOMIZER_URL}/v1/customization/jobs/{job_id}/status",
            timeout=10
        )
        
        if response.status_code != 200:
            print(f"❌ Failed to get initial job status: {response.status_code}")
            print(f"   Response: {response.text[:500]}")
            return None
        
        status_data = response.json()
        job_status = status_data.get('status')
        print(f"\nInitial job status: {job_status}")
        print(f"   Created at: {status_data.get('created_at')}")
        
        # Poll until job completes
        while job_status in ['created', 'pending', 'running']:
            time.sleep(polling_interval)
            
            response = requests.get(
                f"{CUSTOMIZER_URL}/v1/customization/jobs/{job_id}/status",
                timeout=10
            )
            
            if response.status_code == 200:
                status_data = response.json()
                job_status = status_data.get('status')
                elapsed = time.time() - start_time
                
                # Get progress info if available
                progress = status_data.get('percentage_done', 0)
                steps_completed = status_data.get('steps_completed', 0)
                epochs_completed = status_data.get('epochs_completed', 0)
                
                print(f"\nJob status: {job_status} (after {elapsed:.1f}s / {elapsed/60:.1f}m)")
                print(f"   Progress: {progress}%")
                print(f"   Steps: {steps_completed}, Epochs: {epochs_completed}")
                
                # Informational: Check if waiting for GPU (running but no progress)
                if job_status == 'running' and steps_completed == 0 and elapsed > 120:
                    print(f"\nℹ️  Job has been running for {elapsed/60:.1f} minutes with 0% progress")
                    print(f"   This may indicate the training pod is waiting for GPU resources")
                    print(f"   Monitoring will continue - job will proceed once GPU is available")
            else:
                print(f"⚠️  Error checking status: {response.status_code}")
            
            # No timeout - continue monitoring indefinitely
        
        # Final status
        print(f"\n{'='*60}")
        print(f"Final job status: {job_status}")
        print(f"Total time: {(time.time() - start_time)/60:.1f} minutes")
        print(f"{'='*60}")
        
        return job_status
        
    except Exception as e:
        print(f"❌ Error waiting for job: {e}")
        import traceback
        traceback.print_exc()
        return None

if 'job_id' in locals() and job_id:
    job_status = wait_customization_job(job_id=job_id)
    if job_status:
        if job_status == 'completed':
            print(f"\n✅ Customization job completed successfully!")
            print(f"   The customized model should now be available")
        elif job_status == 'failed':
            print(f"\n❌ Customization job FAILED")
            print(f"   Check Customizer logs: oc logs -n {NMS_NAMESPACE} -l app=nemocustomizer --tail=200")
        else:
            print(f"\n⚠️  Customization job finished with status: {job_status}")
    else:
        print(f"\n⚠️  Could not determine job status")
else:
    print("⚠️  No job ID available - skipping wait step")
    print("   Make sure to run the 'Create Customization Job' cell first")
    job_status = None

In [ ]:
# Verify customization job completed successfully
if 'job_status' in locals() and job_status == 'completed':
    print("✅ Customization job completed successfully!")
    print(f"\n📦 Customized model details:")
    if 'unique_output_model' in locals():
        print(f"   Output model: {unique_output_model}")
    elif 'job_response' in locals() and job_response:
        print(f"   Output model: {job_response.get('output_model', 'N/A')}")
    print(f"   Job ID: {job_id if 'job_id' in locals() else 'N/A'}")
    print(f"\n💡 Next steps:")
    print(f"   1. The customized model is now stored in NeMo's model registry")
    print(f"   2. To use it for inference, create a new InferenceService with the customized model")
    print(f"   3. Or download the model artifacts from the Entity Store")
    
    # Try to get job details
    if 'job_id' in locals():
        try:
            response = requests.get(
                f"{CUSTOMIZER_URL}/v1/customization/jobs/{job_id}",
                timeout=10
            )
            if response.status_code == 200:
                job_details = response.json()
                print(f"\n📊 Job details:")
                print(json.dumps(job_details, indent=2))
        except Exception as e:
            print(f"\n⚠️  Could not fetch job details: {e}")
else:
    print("⚠️  Customization job did not complete successfully")
    if 'job_status' in locals():
        print(f"   Job status: {job_status}")
    else:
        print(f"   Job status not available - make sure to run the wait step first")

## Summary

This notebook has tested the complete customization workflow:
1. ✅ Customizer service health (`/v1/health/ready`)
2. ✅ Customization jobs listing (`/v1/customization/jobs`)
3. ✅ DataStore service connectivity (`/v1/health`)
4. ✅ Entity Store service connectivity (`/health`)
5. ✅ DataStore namespace creation
6. ✅ Dataset upload to DataStore (via HuggingFace API)
7. ✅ Dataset registration in Entity Store
8. ✅ Verification that Customizer can find the dataset
9. ✅ Base model testing (before customization)
10. ✅ Training data preparation and upload
11. ✅ Customization dataset registration
12. ✅ Customization job creation
13. ✅ Job monitoring and completion verification

### Key Findings

**DataStore Upload**: ✅ Works via HuggingFace-compatible API (`{DATASTORE_URL}/v1/hf`)

**Entity Store Registration**: ✅ Required for Customizer to find datasets
- Customizer uses Entity Store dataset IDs, not direct DataStore paths
- Files URL format: `hf://datasets/{namespace}/{dataset_name}`

**Customizer Access**: ✅ Can access datasets via Entity Store
- Customizer queries Entity Store to resolve dataset IDs
- Entity Store provides the `hf://datasets/...` URL that points to DataStore

**Customization Workflow**: ✅ Complete end-to-end workflow tested
- Base model testing
- Training data preparation and upload
- Customization job creation and monitoring
- Job completion verification

### Next Steps

After customization completes:
1. The customized model is stored in NeMo's model registry
2. Create a new InferenceService with the customized model for inference
3. Compare responses between base and customized models to verify customization worked

See the [reference notebook](https://github.com/NVIDIA/k8s-nim-operator/blob/69e19c94bb8dcf3003ae553e05303cecb0da1d24/test/e2e/jupyter-notebook/e2e-notebook.ipynb) for more examples.